In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
def add_user_message(messages, text):
    user_message={"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0 ):
    params={
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

# 시스템 메시지가 설정된 경우에만 params에 system을 포함하게끔 하는 if 함수
    if system:
        params["system"]= system
# (**)는 params 주머니 안의 내용물을 자동으로 꺼내서 API 양식에 맞게 채워주는 규칙임     
    message=client.messages.create(**params)
    return message.content[0].text

In [4]:
#실습문제2번에 temperature 변수 추가하여 0일 때와 1일 때 비교하기 
#temperature가 0일 때
messages = []

add_user_message(
    messages,
    "기말 현재 차입금 상환약정을 위반했는데, 은행에서 해당 차입금에 대한 상환요청이 오지 않았다는 이유로 회사는 비유동 유지하고자 희망하는 상황. 해당 차입금에 대해 실증절차 수행 후 감사조서화 필요.")
answer = chat(messages, system="너는 10년 차 감사인이야. 설명은 생략하고 '수행절차' '필수 추가검토자료' '결론'과 '조서용 문구'만 아주 간결하게 답변해.",
             temperature=0.0)

answer

'## 차입금 분류 검토 (기말 약정위반)\n\n---\n\n### 수행절차\n\n1. 차입약정서 입수 → 위반 조항 및 내용 확인\n2. 기말 현재 약정위반 사실 확인 (재무비율, 담보유지, 이자지급 등)\n3. **은행 조회서 발송** → 기말 현재 잔액, 만기, 약정위반 인지 여부, 상환청구 의사 확인\n4. 기말 후 **면제(Waiver) 또는 유예(Grace period) 약정** 체결 여부 확인\n   - 체결 시: 서면 확인, 체결일자 확인 (기말 前/後 구분)\n5. KIFRS 1001 문단 74~76 적용 요건 검토\n\n---\n\n### 필수 추가검토자료\n\n| 자료 | 목적 |\n|------|------|\n| 차입약정서 원본 | 위반 조항 특정 |\n| 약정위반 확인서 (은행) | 위반 사실 객관적 확인 |\n| Waiver letter (있는 경우) | 비유동 유지 가능 여부 판단 |\n| 기말 후 은행 공문/이메일 | 상환청구 의사 부재 확인 |\n| 이사회/경영진 확인서 | 경영진 인식 확인 |\n\n---\n\n### 결론\n\n> **기말 현재 약정위반 → 원칙적으로 유동 분류 (KIFRS 1001 문단 74)**\n>\n> 단, **기말일로부터 보고서 발행일 이전에** 대여자가 서면으로 유예(최소 12개월) 동의한 경우에 한해 비유동 유지 가능.\n>\n> **상환청구가 오지 않았다는 사실만으로는 비유동 분류 근거 불충분.**\n>\n> → 회사 주장 불수용, **유동 재분류 요청**. 미수용 시 수정사항으로 처리.\n\n---\n\n### 조서용 문구\n\n> 기말 현재 차입금(XXX원)에 대해 차입약정 위반 사실이 확인되었으며, KIFRS 1001 문단 74에 따라 동 차입금은 유동부채로 분류되어야 한다. 회사는 기말 현재 대여자로부터 상환청구를 받지 않았음을 근거로 비유동 분류를 주장하나, 동 기준서는 기말 현재 약정위반이 존재하는 경우 대여자의 상환청구 여부와 무관하게 유동 분류를 요구하고 있다. 기말일 이후 보고서

In [5]:
#실습문제2번에 temperature 변수 추가하여 0일 때와 1일 때 비교하기 
#temperature가 1일 때
messages = []

add_user_message(
    messages,
    "기말 현재 차입금 상환약정을 위반했는데, 은행에서 해당 차입금에 대한 상환요청이 오지 않았다는 이유로 회사는 비유동 유지하고자 희망하는 상황. 해당 차입금에 대해 실증절차 수행 후 감사조서화 필요.")
answer = chat(messages, system="너는 10년 차 감사인이야. 설명은 생략하고 '수행절차' '필수 추가검토자료' '결론'과 '조서용 문구'만 아주 간결하게 답변해.",
             temperature=1.0)

answer

'## 차입금 분류 검토 (기말 약정위반)\n\n---\n\n### 수행절차\n\n1. 차입약정서 징구 → 위반 조항 및 내용 확인\n2. 은행 잔액확인서 상 **기한이익상실(EOD) 조항** 명시 여부 확인\n3. 회사 측 위반사실 인지 시점 및 은행과의 커뮤니케이션 내역 확인\n4. **기말일 현재** 은행의 면제(Waiver) 또는 유예(Grace period) 부여 여부 확인\n   - Waiver 존재 시 → 서면 확인서 징구, 기말 이전 부여 여부 판단\n5. K-IFRS 1001호 문단 74~76 요건 충족 여부 검토\n   - 기말일 현재 Waiver 미확보 시 → **유동 재분류 필요**\n6. 경영진 의견 서면 수취 및 반박논거 문서화\n\n---\n\n### 필수 추가검토자료\n\n| 자료 | 목적 |\n|------|------|\n| 차입약정서 원본 | 위반 조항 확정 |\n| 은행 잔액확인서 | EOD 문구 확인 |\n| Waiver 서면 (있을 경우) | 유동/비유동 판단 근거 |\n| 은행 공문·이메일 | 상환요청 부재 증빙 |\n| 경영진 확인서 | 주장 문서화 |\n\n---\n\n### 결론\n\n> K-IFRS 1001.74에 따라, **기말일 현재 채권자의 서면 Waiver가 없는 한** 동 차입금은 **유동부채로 재분류**되어야 함.\n> 단순히 상환요청이 없었다는 사실은 비유동 유지의 근거가 되지 않음.\n> 회사가 재분류를 거부할 경우 → **왜곡표시 금액 집계 후 중요성 판단** 진행.\n\n---\n\n### 조서용 문구\n\n> "기말 현재 당사는 차입약정 상 [○○조항]을 위반하였으며, 기말일 현재 채권자로부터 서면에 의한 채무면제(Waiver) 또는 유예를 확보하지 못한 것으로 확인되었다. K-IFRS 제1001호 문단 74에 의거, 동 차입금 [XXX백만원]은 기말 현재 유동부채로 분류되어야 한다. 채권자의 상환요청 부재는 동 기준서상 비유동 분류의 요건을 충족하지 아니하므로, 회사의 비유동 유지 주장은 수